[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/11_sliding_window.ipynb)

# 🔴 Hard: Sliding Window Attention

Implement **Sliding Window Attention** — used in Longformer, Mistral, etc. for efficient long-context processing.

Each position $i$ can only attend to positions $j$ where $|i - j| \le w$ (the window size).

### Signature
```python
def sliding_window_attention(Q, K, V, window_size):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
    # window_size: int — position i attends to [i-w, i+w]
```

### Rules
- Do **NOT** use sparse attention libraries
- Mask positions outside the window with `-inf`
- `window_size=0`: only self — output should equal V
- `window_size >= seq_len`: equivalent to full attention

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.4 MB/s eta 0:00:00


In [2]:
import torch
import math

In [26]:
# ✏️ YOUR IMPLEMENTATION HERE

def sliding_window_attention(Q, K, V, window_size):
    # pass  # Replace this
    # score = torch.bmm(Q, K.transpose(-2, -1)) / math.sqrt(Q.shape[-1]) # (B, T, d_k)
    # mask = torch.ones_like(score)
    # upper = torch.triu(mask, diagonal = -window_size).bool()
    # lower = torch.tril(mask, diagonal = window_size).bool()
    # mask = (upper & lower)
    # score = score.masked_fill(~mask, -float('inf'))  # make sure to inver the mask
    # att_w = torch.softmax(score, dim =-1 )
    # return torch.bmm(att_w, V)

    # Better solution:
    B, T, d_k = Q.shape
    score = torch.bmm(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

    i = torch.arange(T, device = Q.device).unsqueeze(1)
    # column vector (6, 1)
    # [[0],
    #  [1],
    #  [2],
    #  [3],
    #  [4],
    #  [5]]

    j = torch.arange(T, device = Q.device).unsqueeze(0)
    # row vector (1, 6)
    # [[0, 1, 2, 3, 4, 5]]

    mask = (torch.abs(i-j) > window_size).bool()
    score = score.masked_fill(mask, -float('inf'))
    att_w = torch.softmax(score, dim=-1)
    return torch.bmm(att_w, V)

In [27]:
# 🧪 Debug
Q = torch.randn(1, 6, 8)
K = torch.randn(1, 6, 8)
V = torch.randn(1, 6, 8)

out = sliding_window_attention(Q, K, V, window_size=1)
print("Output shape:", out.shape)  # (1, 6, 8)

# window=0 should return V
out0 = sliding_window_attention(Q, K, V, window_size=0)
print("window=0 == V?", torch.allclose(out0, V, atol=1e-5))

Output shape: torch.Size([1, 6, 8])
window=0 == V? True


In [28]:
from torch_judge import check
check('sliding_window')


🧪 Testing: Sliding Window Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (1.7ms)
  ✅ [2/5] window_size=0 — only sees itself (0.5ms)
  ✅ [3/5] Large window equals full attention (3.0ms)
  ✅ [4/5] Distant tokens don't affect output (2.0ms)
  ✅ [5/5] Gradient flow (0.8ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (8.1ms total)
  Progress saved. Run status() to see your dashboard.

